# Part 7 — State Estimation and SLAM

State estimation fuses noisy sensor measurements over time to answer: where am I and what is around me?

**Learning style:** mechanisms first → frameworks second → real systems third. The notebook is intentionally slow, explicit, and beginner-friendly.

In [ ]:
# Setup: run this first.
# Works from the repository root. In Colab, clone the repo first, then run from inside it.
from pathlib import Path
import sys, math, random
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    print('Tip: run this notebook from the repository root, or clone the repo in Colab first.')
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

## 1. Mental model

State estimation fuses noisy sensor measurements over time to answer: where am I and what is around me?

Before code, write one sentence in your own words: *what problem does this topic solve?*

## 2. Mechanism and math

A Kalman filter alternates prediction and correction:
\[
\hat{x}_{t|t-1}=A\hat{x}_{t-1}+Bu_t
\]
\[
\hat{x}_{t}=\hat{x}_{t|t-1}+K(z_t-H\hat{x}_{t|t-1})
\]
Prediction trusts the motion model; correction trusts sensors according to uncertainty.

## 3. From-scratch lab

Track 1D position/velocity from noisy position measurements.

Read every line. The code avoids clever abstractions so you can see the mechanism.

In [ ]:
import numpy as np
from robotics.filters import KalmanFilter
A = np.array([[1,1],[0,1]], float)
H = np.array([[1,0]], float)
Q = np.eye(2)*0.01
R = np.array([[0.25]])
kf = KalmanFilter(A=A, H=H, Q=Q, R=R, x=np.array([0,1.0]), P=np.eye(2))
for z in [1.2, 1.9, 3.1, 3.8, 5.2]:
    kf.predict()
    kf.update(np.array([z]))
    print('measurement', z, 'state estimate', np.round(kf.x, 2))

## 3.1 Code reading guide

When you read the previous cell, do not treat it as a black box. Trace it in this order:

1. **Inputs:** what are the given numbers, observations, states, rewards, or measurements?
2. **Internal variables:** what does each variable represent physically or mathematically?
3. **Update rule:** which line is the core mechanism from the math section?
4. **Output:** what should change if the mechanism is working?
5. **Failure case:** what parameter could make the example unstable, wrong, or unsafe?

This habit is the bridge between toy examples and real robotics code: every simulator, ROS node, policy, controller, or perception model still has inputs, state, an update rule, and outputs.

## 4. Framework/practice view

Frameworks: robot_localization in ROS, GTSAM for factor graphs, Cartographer/RTAB-Map for SLAM, PX4 EKF2 for drones.

The goal is not to replace understanding with APIs. The goal is to recognize the same mechanism when a library hides the details.

In [ ]:
print('Framework practice: inspect robot_localization inputs: IMU, odometry, GPS, wheel encoders.')

## 4.1 Framework comparison checklist

After running or reading the framework cell, write a small mapping table for yourself:

| Question | Your answer |
|---|---|
| What object/function in the framework replaces the scratch code? |  |
| Which parameters match the math symbols? |  |
| What details does the framework hide? |  |
| What new engineering concerns appear? | installation, devices, logging, data formats, batching, safety, versioning |

This is where top-down learning becomes useful: you learn the professional API **without losing the mechanism**.

## 5. Real-system connection

A planner needs a belief, not just raw sensors. SLAM closes the loop by estimating both robot trajectory and map.

## 6. Exercises

1. Increase measurement noise R.
2. Increase process noise Q.
3. What sensors would you fuse for a car vs a drone?

**Notebook habit:** after each exercise, add a short note explaining what changed and why it matters in a robot/car/drone/VLA stack.